In [19]:
import src.database.scripts.sql as sql
from src.database.data_collector.recipes_fetch import recipes_parser

In [ ]:
def drop_recipes_table():
    query = "DROP TABLE IF EXISTS recipes"
    cursor.execute(query)

def create_recipes_table():
    query = """
    CREATE TABLE IF NOT EXISTS recipes(
    recipe_id   INT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    name        TEXT,
    amount      INT,
    rarity      TEXT,
    merchant    TEXT
    )"""
    cursor.execute(query)

def recipes_fetch():
    praser = recipes_parser()
    recipes_list = []
    for table in praser.table_range():
        print(f'parsing table:{table}', end='\r')
        for row in praser.row_range(table):
            praser.row_target(table, row)
            amount, rarity, name = praser.item()
            merchant = praser.merchant()[0]
            recipes_list.append((name, amount, rarity, merchant))
    return recipes_list

def insert_recipes_table():
    row_list = recipes_fetch() 
    query = """
    INSERT INTO recipes (name, amount, rarity, merchant)
    VALUES (%s, %s, %s, %s)
    """
    cursor.executemany(query, row_list)

In [49]:
def drop_ingredients_table():
    query = "DROP TABLE IF EXISTS ingredients"
    cursor.execute(query)

def create_ingredients_table():
    query = """
    CREATE TABLE IF NOT EXISTS ingredients(
    recipe_id   INT NOT NULL REFERENCES recipes(recipe_id),
    name        TEXT,
    amount      INT,  
    rarity      TEXT,
    PRIMARY KEY (recipe_id, name, rarity)
    )"""
    cursor.execute(query)

def ingredients_fetch():
    praser = recipes_parser()
    ingredients_list = []
    recipe_id = 1
    for table in praser.table_range():
        print(f'parsing table:{table}', end='\r')
        for row in praser.row_range(table):
            praser.row_target(table, row)
            ingredients = [(recipe_id,) + i for i in praser.ingredients()]
            ingredients_list.extend(ingredients)
            recipe_id += 1
    return ingredients_list

def insert_ingredients_table():
    ingredients_list = ingredients_fetch()
    query = """
    INSERT INTO ingredients (recipe_id, amount, rarity, name)
    VALUES (%s, %s, %s, %s)
    """
    cursor.executemany(query, ingredients_list)

def amount_correction():
    query = """
    UPDATE recipes
    SET amount = 3
    WHERE name LIKE '%Potion%'
    OR name = 'Poison Vial'
    OR name = 'Ghostdust Pouch'
    """
    cursor.execute(query)

In [50]:
def recipe_creation():
    if __name__ == "__main__":
        conn = sql.connect_pc()
        cursor = conn.cursor()

        drop_recipes_table()
        create_recipes_table()
        insert_recipes_table()

        conn.commit()
        conn.close()

def ingredients_creation():
    if __name__ == "__main__":
        conn = sql.connect_pc()
        cursor = conn.cursor()

        drop_ingredients_table()
        create_ingredients_table()
        ingredients_fetch()
        insert_ingredients_table()
        amount_correction()

        conn.commit()
        conn.close()

In [54]:
if __name__ == "__main__":
    conn = sql.connect_pc()
    cursor = conn.cursor()

    drop_ingredients_table()
    drop_recipes_table()

    recipe_creation()
    ingredients_creation()

    conn.commit()
    conn.close()